# CataList Media | AMI Meeting Memory Intelligence Agent
### Agentic AI Intern Build Test: Multi-Meeting Memory & Verification Engine

This notebook demonstrates the complete end-to-end memory engine across **138 AMI meeting transcripts (~676,000 words, 48,148 dialogue turns)**:
1. **Zero-API Ingestion & Diarization:** SQLite FTS5 + 384-dimensional dense embeddings stored locally.
2. **Fast Cloud Synthesis:** Uses **Gemini 3.5 Flash Lite** via Google GenAI Interactions API with deterministic Pydantic schema guardrails.
3. **Deterministic Verification Guardrail:** Verifies all citations word-for-word against raw transcripts with character-level alignment.
4. **Team TS3012 Executive Briefing:** Synthesized strictly from `TS3012a-c` with historic quarantine of `TS3012d`.

## 1. Environment Setup & Dependencies
Install required lightweight dependencies. Zero heavy frameworks (no LangChain/CrewAI) to guarantee deterministic execution.

In [ ]:
!pip install -q python-dotenv google-genai rapidfuzz

# Set your Gemini API key
import os
os.environ["GEMINI_API_KEY"] = "" # Enter Gemini API Key here
os.environ["GEMINI_MODEL"] = "gemini-3.5-flash-lite"

## 2. Database & Retriever Health Check
Verify the pre-built SQLite relational database (`data/meeting_store.db`).

In [ ]:
import sqlite3
conn = sqlite3.connect("data/meeting_store.db")
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM turns")
total_turns = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM meetings")
total_meetings = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM turn_embeddings")
total_embeddings = cur.fetchone()[0]
conn.close()

print(f"Database Health Check:")
print(f"  - Total Meetings Indexed: {total_meetings} (100% of AMI Corpus)")
print(f"  - Total Dialogue Turns:   {total_turns:,}")
print(f"  - Stored BLOB Embeddings: {total_embeddings:,}")

## 3. Run the 2-Step Agent on Official Evaluation Questions
Demonstrates dynamic retrieval, dialogue gap bridging, and deterministic verification.

In [ ]:
from src.agent import MeetingAgent
from src.verifier import CitationVerifier

agent = MeetingAgent()
verifier = CitationVerifier()

# Test Q01 (Team TS3006 Feature Drop)
q1 = {
    "question_id": "Q01",
    "team": "TS3006",
    "question": "Team TS3006 planned both an LCD screen and speech recognition. When the costs forced a cut, which of the two did they drop, which did they keep, and what reason was given for keeping it?"
}

res = agent.answer_question(q1, verbose=True)
print("\n--- SYNTHESIZED ANSWER ---")
print(res["answer"])
print("\n--- VERIFIED VERBATIM CITATIONS ---")
for c in res["citations"]:
    is_val, match, _ = verifier.verify_quote(c["meeting_id"], c["quote"])
    status = "VERIFIED VERBATIM" if is_val else "UNVERIFIED"
    print(f"  [{c['meeting_id']}] [{status}]: \"{c['quote']}\"")

## 4. Generate TS3012 Executive Briefing Memo (Strict Quarantine of `TS3012d`)
Synthesizes decisions, changes, open action items, and unresolved tensions from `TS3012a-c` while strictly barring `TS3012d`.

In [ ]:
from src.brief import BriefGenerator

brief_engine = BriefGenerator()
brief_text = brief_engine.generate_brief(client=agent.client)
print(brief_text)

## 5. Interactive QA Explorer
Ask any plain-English question across all 138 meetings.

In [ ]:
user_q = "What material did Team ES2003 choose for the remote casing?"
team_extracted = agent.retriever.extract_team_id(user_q)
ans = agent.answer_question({"question_id": "INTERACTIVE", "team": team_extracted, "question": user_q})

print(f"Query: {user_q}")
print(f"Detected Scope: {team_extracted or 'Global'}")
print(f"Answer: {ans['answer']}")
for c in ans["citations"]:
    print(f"  Evidence: [{c['meeting_id']}] \"{c['quote']}\"")